# Presentación 1 - Notebook reproducible

Predicción de **energía mensual facturada (kWh)** por comuna / tipo de cliente / tarifa en Chile.

**Objetivo del notebook**: permitir replicar todas las conclusiones de la Presentación 1 desde cero.

Este notebook está pensado para ejecutarse **desde la raíz del repositorio** (`/proyecto_DS/`).

- Dataset principal: `data/interim/facturacion_clean.parquet`
- Dataset auxiliar: `data/processed/modeling_con_auxiliares.parquet`
- Reportes asociados (todos en `reports/`):
  - `auditoria_datos.md`
  - `calidad_target_energia.md`
  - `resumen_analisis_exploratorio.md`
  - `resumen_modelo_energia.md`
  - `seleccion_modelo_y_correlacion.md`
  - `guion_presentacion.md`

> Regla del proyecto: **no se modifican** archivos en `data/raw/` ni `data/interim/`. Cualquier filtro se aplica sólo dentro del pipeline.


## 2. Importación de librerías

In [ ]:
from __future__ import annotations

from pathlib import Path
import subprocess
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

warnings.filterwarnings('ignore', category=UserWarning)

# Detectar raíz del repositorio (funciona desde notebooks/ o desde la raíz)
REPO = Path.cwd()
if not (REPO / 'data').exists() and (REPO.parent / 'data').exists():
    REPO = REPO.parent

BASE_PATH = REPO / 'data' / 'interim' / 'facturacion_clean.parquet'
AUX_PATH  = REPO / 'data' / 'processed' / 'modeling_con_auxiliares.parquet'

# Regenerar datos intermedios si no existen (pipeline desde cero)
if not BASE_PATH.exists():
    print('Generando datos intermedios...')
    r = subprocess.run(['python', str(REPO / 'src' / '02_limpieza_facturacion.py')],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'Error en limpieza:\n{r.stderr}')
    print('Limpieza completada.')

if not AUX_PATH.exists():
    print('Generando datasets de modelado...')
    for script in ['04_preparar_datasets_modelado.py', '05_integrar_auxiliares.py']:
        r = subprocess.run(['python', str(REPO / 'src' / script)],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(f'Error en {script}:\n{r.stderr}')
    print('Datasets generados.')

print('REPO root:', REPO)
print('Base exists:', BASE_PATH.exists())
print('Aux  exists:', AUX_PATH.exists())


## 3. Carga de datos

In [ ]:
df = pd.read_parquet(BASE_PATH)
df["fecha"] = pd.to_datetime(df["fecha"])
print("shape:", df.shape)
df.head(3)


## 4. Descripción del dataset

Columnas relevantes:

- `fecha` (datetime, mensual desde 2015-01-01 hasta 2024-12-01)
- `anio`, `mes` (enteros)
- `region`, `comuna`, `tipo_clientes`, `tarifa` (categóricas)
- `clientes_facturados` (numérica)
- `e1_kwh`, `e2_kwh`, `energia_kwh` (numéricas; `energia_kwh ≈ e1 + e2`)
- `consumo_promedio_cliente_kwh` (derivada)

**Target del proyecto**: `energia_kwh` (regresión sobre panel temporal mensual).

In [ ]:
print("Cols:", list(df.columns))
print()
print("Tipos:")
print(df.dtypes)
print()
print("Rango fecha:", df["fecha"].min().date(), "→", df["fecha"].max().date())
print("Meses únicos:", df["fecha"].nunique())
print("Filas:", len(df))


## 5. Preparación de la vista de modelado

Se aplica un filtro **dentro del pipeline** (no se modifica `clean`):

```
vista = clean[(clean.energia_kwh > 0) & (clean.clientes_facturados > 0)]
```

Cohorte temporal: 2018-04-01 → 2024-12-01 (idéntica al modelo v2 para comparabilidad).

In [ ]:
COHORT_START = pd.Timestamp("2018-04-01")
COHORT_END = pd.Timestamp("2024-12-01")

cohort = df[(df["fecha"] >= COHORT_START) & (df["fecha"] <= COHORT_END)].copy()
view = cohort[(cohort["energia_kwh"] > 0) & (cohort["clientes_facturados"] > 0)].copy()

print(f"Filas en cohorte:     {len(cohort):,}")
print(f"Filas en vista:       {len(view):,}")
print(f"Reducción por filtro: {len(cohort) - len(view):,} filas eliminadas")


## 6. Aclaración: ceros del target vs nulos de lags

> Esto es **crítico** para no confundir dos fenómenos distintos:

- **Ceros de `energia_kwh`** son valores reales que ya existen en el dataset
  original (13 283 filas en `clean`, ≈2.7 %). Provienen de combinaciones
  geografía × tarifa que en ese mes no tuvieron consumo (o reporte 0).
  **NO son producidos por generar lags.**
- **Nulos en columnas `_L1`, `_L2`, `_L3`** vienen del diseño de los lags:
  los primeros meses de la serie no tienen historia previa, por lo que
  `shift(1/2/3)` produce `NaN`.
- En escenarios con auxiliares (modelo v2 escenario D), las filas sin
  lags válidos se eliminan explícitamente con
  `dropna(subset=lag_cols)`.

Verificamos los ceros del target abajo, **sin construir lags**, para que
quede claro que existen en `clean` por sí solos.

In [ ]:
n_zero = int((df["energia_kwh"] == 0).sum())
n_neg = int((df["energia_kwh"] < 0).sum())
n_pos = int((df["energia_kwh"] > 0).sum())
print(f"energia_kwh == 0 : {n_zero:6d} filas ({n_zero/len(df):.3%})")
print(f"energia_kwh <  0 : {n_neg:6d} filas")
print(f"energia_kwh >  0 : {n_pos:6d} filas")


## 7. EDA básico

Para profundizar, ver:

- `reports/figures_v2/01_evolucion_mensual_nacional.png`
- `reports/figures_v2/03_energia_por_region.png`
- `reports/figures_v2/05_boxplot_tipo_cliente.png`
- `reports/resumen_analisis_exploratorio.md`

Aquí mostramos sólo dos visualizaciones rápidas para situar al lector.

In [ ]:
KWH_TO_TWH = 1e9
agg = (
    view.groupby("fecha")["energia_kwh"].sum() / KWH_TO_TWH
).reset_index().rename(columns={"energia_kwh": "energia_twh"})

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(agg["fecha"], agg["energia_twh"], color="#4477AA", linewidth=1.5)
ax.set_title("Energía total facturada por mes (TWh, cohorte 2018-04 → 2024-12)")
ax.set_ylabel("TWh"); ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()


In [ ]:
by_tipo = view.groupby("tipo_clientes")["energia_kwh"].sum() / KWH_TO_TWH
print("Energía total por tipo (TWh):")
print(by_tipo.round(1))
ratio = by_tipo.max() / by_tipo.min()
print(f"Ratio mayor/menor: {ratio:.2f}")


## 8. Correlation heatmap

Se construye sobre **variables numéricas** únicamente: `energia_kwh`, `clientes_facturados`, `anio`, `mes`. Las categóricas (`region`, `comuna`, `tarifa`, `tipo_clientes`) **no se reflejan aquí**; su efecto se mide indirectamente vía OneHotEncoder dentro del modelo.

In [ ]:
num_cols = ["energia_kwh", "clientes_facturados", "anio", "mes"]
corr_num = view[num_cols].corr(method="pearson")
print("Pearson (raw):")
print(corr_num.round(3))

vlog = view[num_cols].copy()
vlog["log_energia_kwh"] = np.log1p(vlog["energia_kwh"])
vlog["log_clientes_facturados"] = np.log1p(vlog["clientes_facturados"])
corr_log = vlog[["log_energia_kwh", "log_clientes_facturados", "anio", "mes"]].corr()
print()
print("Pearson (log-log):")
print(corr_log.round(3))


In [ ]:
def heatmap(corr, title):
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr.columns)))
    ax.set_xticklabels(corr.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(corr.columns)))
    ax.set_yticklabels(corr.columns)
    for i in range(len(corr.columns)):
        for j in range(len(corr.columns)):
            ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center",
                    color="white" if abs(corr.values[i, j]) > 0.5 else "black", fontsize=9)
    fig.colorbar(im, ax=ax)
    ax.set_title(title)
    fig.tight_layout()
    plt.show()

heatmap(corr_num, "Heatmap Pearson (raw)")
heatmap(corr_log, "Heatmap Pearson (log-log)")


## 9. Split temporal

Holdout por fecha única, **80/20** de los meses. Verificamos:

- `max(fecha_train) < min(fecha_test)`
- meses train y test son disjuntos

In [ ]:
unique_dates = sorted(view["fecha"].unique())
n_train = int(len(unique_dates) * 0.8)
train_dates = set(unique_dates[:n_train])
test_dates = set(unique_dates[n_train:])

assert max(train_dates) < min(test_dates), "split temporal viola orden"
assert not (train_dates & test_dates), "hay meses compartidos entre train y test"

train = view[view["fecha"].isin(train_dates)].copy()
test = view[view["fecha"].isin(test_dates)].copy()

print(f"Train: {min(train_dates).date()} → {max(train_dates).date()} | meses={len(train_dates)} | filas={len(train):,}")
print(f"Test : {min(test_dates).date()}  → {max(test_dates).date()}  | meses={len(test_dates)} | filas={len(test):,}")


## 10. Comparación de 7 modelos

| Modelo | Tipo | Notas |
|---|---|---|
| `DummyRegressor(mean)` | baseline | predice la media de train |
| `LinearRegression` | lineal | OHE + StandardScaler |
| `Ridge(α=1)` | lineal regularizado | OHE + StandardScaler |
| `DecisionTreeRegressor(d=10)` | árbol | OHE sin escalar |
| `RandomForestRegressor(n=100, d=10)` | ensamble bagging | OHE sin escalar |
| `ExtraTreesRegressor(n=100, d=10)` | ensamble extremo | OHE sin escalar |
| `KNeighborsRegressor(k=5)` | basado en distancia | OHE + StandardScaler, submuestra de train para evitar O(n²) |

In [ ]:
CAT = ["region", "comuna", "tipo_clientes", "tarifa"]
NUM = ["anio", "mes", "clientes_facturados"]
FEATURES = NUM + CAT
TARGET = "energia_kwh"
RNG = 42

def preprocessor(scale: bool):
    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(num_steps), NUM),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", drop="first")),
        ]), CAT),
    ], remainder="drop")

X_tr = train[FEATURES]; y_tr = train[TARGET]
X_te = test[FEATURES];  y_te = test[TARGET]

models = [
    ("DummyRegressor(mean)",       Pipeline([("m", DummyRegressor(strategy="mean"))]),                                              False),
    ("LinearRegression",            Pipeline([("pre", preprocessor(True)),  ("m", LinearRegression())]),                              False),
    ("Ridge(α=1)",                  Pipeline([("pre", preprocessor(True)),  ("m", Ridge(alpha=1.0))]),                                False),
    ("DecisionTree(d=10)",          Pipeline([("pre", preprocessor(False)), ("m", DecisionTreeRegressor(max_depth=10, random_state=RNG))]),               False),
    ("RandomForest(n=100,d=10)",    Pipeline([("pre", preprocessor(False)), ("m", RandomForestRegressor(n_estimators=100, max_depth=10, random_state=RNG, n_jobs=-1))]), False),
    ("ExtraTrees(n=100,d=10)",      Pipeline([("pre", preprocessor(False)), ("m", ExtraTreesRegressor(n_estimators=100, max_depth=10, random_state=RNG, n_jobs=-1))]),  False),
    ("KNN(k=5,scaled)",             Pipeline([("pre", preprocessor(True)),  ("m", KNeighborsRegressor(n_neighbors=5, n_jobs=-1))]),  True),
]

rng = np.random.default_rng(RNG)
idx_knn = rng.choice(len(train), size=min(50_000, len(train)), replace=False)

results = []
for name, pipe, knn_subsample in models:
    if knn_subsample:
        pipe.fit(X_tr.iloc[idx_knn], y_tr.iloc[idx_knn])
    else:
        pipe.fit(X_tr, y_tr)
    pred = pipe.predict(X_te)
    mae = mean_absolute_error(y_te, pred)
    mse = mean_squared_error(y_te, pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_te, pred)
    mape = mean_absolute_percentage_error(y_te, pred)
    results.append({"modelo": name, "MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2, "MAPE": mape})

resdf = pd.DataFrame(results).sort_values(by=["R2", "RMSE", "MAE"], ascending=[False, True, True]).reset_index(drop=True)
resdf.round({"MAE": 0, "RMSE": 0, "R2": 4, "MAPE": 3})


## 11. Selección del modelo

Regla de desempate: si dos modelos tienen R² indistinguible (|ΔR²| < 0.001), elegimos el de **menor MAE** (error absoluto medio más bajo es más interpretable y más estable para reportar a la audiencia).

En este experimento ExtraTrees y RandomForest están dentro de 0.001 puntos de R²; **RandomForest** gana por MAE significativamente menor (≈ −12 000 kWh sobre ExtraTrees).

`accuracy` **NO aplica**: este problema es de regresión, no de clasificación.

In [ ]:
top_r2 = float(resdf["R2"].iloc[0])
tied = resdf[(resdf["R2"] >= top_r2 - 0.001) & (resdf["modelo"] != "DummyRegressor(mean)")]
tied_sorted = tied.sort_values(by="MAE", ascending=True)
best_name = tied_sorted.iloc[0]["modelo"]
print("Top R² =", round(top_r2, 4))
print("Empate(s) dentro de 0.001:", len(tied))
print("Ganador por menor MAE:", best_name)
tied_sorted.round({"MAE": 0, "RMSE": 0, "R2": 4, "MAPE": 3})


## 12. Predicción vs valor real (mejor modelo)

In [ ]:
best_pipe = dict([(n, p) for n, p, _ in models])[best_name]
y_pred = best_pipe.predict(X_te)

fig, ax = plt.subplots(figsize=(6, 5))
sample = min(20000, len(y_te))
rng2 = np.random.default_rng(RNG)
sidx = rng2.choice(len(y_te), size=sample, replace=False)
yt_s = np.asarray(y_te)[sidx]; yp_s = np.asarray(y_pred)[sidx]
ax.scatter(yt_s, yp_s, s=4, alpha=0.25)
lo, hi = min(yt_s.min(), yp_s.min()), max(yt_s.max(), yp_s.max())
ax.plot([lo, hi], [lo, hi], color="red", linewidth=1)
ax.set_xscale("symlog"); ax.set_yscale("symlog")
ax.set_xlabel("energía real (kWh, symlog)")
ax.set_ylabel("energía predicha (kWh, symlog)")
ax.set_title(f"Predicción vs real - {best_name} (muestra {sample})")
ax.grid(alpha=0.3); fig.tight_layout(); plt.show()


## 13. Análisis de residuos

In [ ]:
resid = np.asarray(y_te) - np.asarray(y_pred)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(np.sign(resid) * np.log1p(np.abs(resid)), bins=80, color="#4477AA", alpha=0.85)
axes[0].set_title("Residuos (signed log1p)"); axes[0].grid(alpha=0.3)

axes[1].scatter(yp_s, np.asarray(y_te)[sidx] - yp_s, s=4, alpha=0.25)
axes[1].axhline(0, color="red", linewidth=1)
axes[1].set_xscale("symlog")
axes[1].set_xlabel("predicción (symlog)")
axes[1].set_ylabel("residual")
axes[1].set_title("Residuos vs predicción"); axes[1].grid(alpha=0.3)
fig.tight_layout(); plt.show()

print(f"\nResid - media: {resid.mean():.0f} kWh")
print(f"Resid - mediana: {np.median(resid):.0f} kWh")
print(f"Resid - std:    {resid.std():.0f} kWh")
print(f"Resid - p90 abs: {np.percentile(np.abs(resid), 90):.0f} kWh")


## 14. Análisis de `clientes_facturados` como variable dominante

Evidencia cuantitativa de su dominancia:

- Correlación Pearson cruda con `energia_kwh`.
- Correlación Spearman.
- Versión log-log.
- R² de regresión lineal univariada usando **sólo** `clientes_facturados`.

Comparación con el modelo v2:

- A. RF con `clientes_facturados`: R² ≈ 0.97
- B. RF sin `clientes_facturados`: R² ≈ 0.80
- Diferencia ΔR² ≈ 0.17

**Veredicto**: no es leakage temporal estricto (no usa información futura), pero es una variable **casi contemporánea**. Para predicción honesta del mes siguiente habría que usar su lag (`clientes_facturados_L1`).

In [ ]:
pe = view[["clientes_facturados", "energia_kwh"]].corr(method="pearson").iloc[0, 1]
sp = view[["clientes_facturados", "energia_kwh"]].corr(method="spearman").iloc[0, 1]
v = view.copy()
v["log_e"] = np.log1p(v["energia_kwh"]); v["log_c"] = np.log1p(v["clientes_facturados"])
pl = v[["log_c", "log_e"]].corr().iloc[0, 1]
sl = v[["log_c", "log_e"]].corr(method="spearman").iloc[0, 1]

simple = LinearRegression().fit(X_tr[["clientes_facturados"]], y_tr)
r2_simple = r2_score(y_te, simple.predict(X_te[["clientes_facturados"]]))

print(f"Pearson raw     : {pe:.3f}")
print(f"Spearman raw    : {sp:.3f}")
print(f"Pearson log-log : {pl:.3f}")
print(f"Spearman log-log: {sl:.3f}")
print(f"R² univariado  (solo clientes_facturados → energia_kwh): {r2_simple:.4f}")


In [ ]:
sample_n = min(20000, len(v))
sidx2 = np.random.default_rng(RNG).choice(len(v), size=sample_n, replace=False)
vs = v.iloc[sidx2]

fig, ax = plt.subplots(figsize=(7, 5))
for tipo, color, marker in [("Residencial", "#4477AA", "o"), ("No Residencial", "#EE6677", "x")]:
    sub = vs[vs["tipo_clientes"] == tipo]
    ax.scatter(sub["log_c"], sub["log_e"], s=5, alpha=0.25, label=tipo, color=color, marker=marker)
ax.set_xlabel("log(1 + clientes_facturados)")
ax.set_ylabel("log(1 + energia_kwh)")
ax.set_title(f"clientes_facturados ↔ energia_kwh (log-log)  |  Pearson(log)={pl:.3f}")
ax.legend(); ax.grid(alpha=0.3); fig.tight_layout(); plt.show()


## 15. Feature importance del mejor modelo

In [ ]:
# best_pipe ya está ajustado; recuperamos nombres post-OHE
pre = best_pipe.named_steps["pre"]
mdl = best_pipe.named_steps["m"]
names = []
for n, t, cols in pre.transformers_:
    if n == "num":
        names.extend(cols)
    elif n == "cat":
        names.extend(t.named_steps["ohe"].get_feature_names_out(cols))

imp = mdl.feature_importances_
fi = pd.DataFrame({"feature": names, "importance": imp}).sort_values("importance", ascending=False)
top = fi.head(20).iloc[::-1]

fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(top["feature"], top["importance"])
ax.set_title(f"Top 20 feature importances - {best_name}")
ax.set_xlabel("importance"); ax.grid(axis="x", alpha=0.3)
fig.tight_layout(); plt.show()

fi.head(15)


## 16. Conclusiones

- El target `energia_kwh` puede predecirse a nivel comuna × tarifa × tipo con
  un modelo basado en árboles entrenado sobre 64 meses (2018-04 → 2023-07)
  y validado en 17 meses fuera de muestra (2023-08 → 2024-12).
- El mejor modelo es **RandomForest(n=100, d=10)**, con R² ≈ 0.97 y MAE ≈ 127 000 kWh en el test temporal.
- El R² alto está fuertemente apoyado en `clientes_facturados`, que es
  estructuralmente casi-contemporánea con el target. Sin esa variable, el
  R² del mismo modelo cae a ≈ 0.80, lo que mide poder predictivo
  independiente.
- Auxiliares SEN (lags L1/L2/L3) no mejoraron el modelo en esta entrega
  (son agregados nacionales, no varían intra-mes a nivel comuna).
- El heatmap numérico **no** captura el efecto de region/comuna/tarifa
  porque son categóricas; su efecto está dentro del OneHotEncoder.

## 17. Limitaciones y próximos pasos

**Limitaciones declaradas honestamente:**

- Sin tuning. No se exploraron hiperparámetros sistemáticamente.
- Sólo un holdout temporal único 80/20.
- `clientes_facturados` casi contemporáneo con el target → R² inflado.
- Heatmap no incluye categóricas.
- KNN entrenado sobre submuestra del train; su comparación es informativa.
- 5 275 filas duplicadas sobre la clave panel asumida (≈1 % de las filas) — limitación del dataset original, no resuelta aquí.

**Próximos pasos (entrega 2 o posterior):**

1. Implementar `clientes_facturados_L1` y reevaluar con TimeSeriesSplit.
2. Investigar la dimensión faltante que explica los 4 990 grupos no únicos en clave panel.
3. Comparar con XGBoost / LightGBM (no incluidos aquí).
4. Construir un test verdaderamente fuera de distribución (años 2025+) cuando esté disponible.
5. Estudiar la inestabilidad numérica de Ridge en el escenario con auxiliares (probable shift de distribución).

> Lo que **no sé**:
>
> - Cuál es exactamente la dimensión faltante en la clave panel (¿distribuidora? ¿concesión?).
> - Si los `energia_kwh < 0` corresponden mayormente a refacturaciones o a inyección neta por generación distribuida; faltan datos operativos.
> - URL canónica de la fuente original de `facturacion_clean`.